In [23]:
import torch
import random
import math
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Special token과 기본 설정
PAD = 0
BOS = 21 #Begin of Sentence
EOS = 22

vocab_size = 23 # 0~22
max_len = 8

# 랜덤 데이터 1개 생성
## 입력 시퀀스를 랜덤하게 만들고, 정답은 뒤집은 시퀀스 만듦.

def generate_sample(min_seq_len=3, max_seq_len=5):
    seq_len = random.randint(min_seq_len, max_seq_len)
    seq = [random.randint(1,20) for _ in range(seq_len)]

    src = seq
    tgt = list(reversed(seq))

    return src,tgt
src,tgt = generate_sample()
print("입력:", src)
print("출력:", tgt)
# 여러번 실행해보면 매번 다른 데이터 나옴

Using device: cpu
입력: [6, 17, 9, 11, 9]
출력: [9, 11, 9, 17, 6]


In [24]:
# Special token 붙이기
## encoder 입력: src + [EOS]
## decoder 입력: [BOS] + tgt
## decoder 출력: tgt + [EOS]

def add_special_tokens(src,tgt):
    src_ids = src + [EOS]
    dec_input = [BOS] + tgt
    dec_target = tgt + [EOS]
    return src_ids, dec_input, dec_target
src,tgt = generate_sample()
src_ids, dec_input, dec_target = add_special_tokens(src,tgt)

print("원본 입력:", src)
print("원본 정답:", tgt)
print("encoder 입력:", src_ids)
print("decoder 입력:", dec_input)
print("decoder 출력:", dec_target)

원본 입력: [9, 14, 7]
원본 정답: [7, 14, 9]
encoder 입력: [9, 14, 7, 22]
decoder 입력: [21, 7, 14, 9]
decoder 출력: [7, 14, 9, 22]


In [25]:
# Padding 함수 만들기
def pad_sequence(seq, max_len, pad_value=PAD):
    return seq + [pad_value] * (max_len - len(seq))

example = [3,7,2,EOS]
print("패딩 전:", example)
print("패딩 후:", pad_sequence(example, max_len))

패딩 전: [3, 7, 2, 22]
패딩 후: [3, 7, 2, 22, 0, 0, 0, 0]


In [26]:

# 랜덤 batch 만들기
## 학습할 때, 매번 새로운 랜덤 데이터를 여러개 모아 batch 만듦
def make_batch(batch_size=32, min_seq_len=3, max_seq_len=5):
    src_batch = []
    dec_input_batch = []
    dec_target_batch = []

    for _ in range(batch_size):
        src,tgt = generate_sample(min_seq_len, max_seq_len)
        src_ids, dec_input, dec_target = add_special_tokens(src,tgt)

        src_batch.append(pad_sequence(src_ids, max_len))
        dec_input_batch.append(pad_sequence(dec_input, max_len))
        dec_target_batch.append(pad_sequence(dec_target, max_len))
    src_batch = torch.tensor(src_batch, dtype=torch.long, device=device)
    dec_input_batch = torch.tensor(dec_input_batch, dtype=torch.long, device=device)
    dec_target_batch = torch.tensor(dec_target_batch, dtype=torch.long, device=device)
    return src_batch,dec_input_batch,dec_target_batch

src_batch,dec_input_batch,dec_target_batch = make_batch(batch_size=4)

print("src_batch shape:", src_batch.shape)
print(src_batch)
print()
print("dec_input_batch", dec_input_batch.shape)
print(dec_input_batch)
print()
print("dec_target_batch", dec_target_batch.shape)
print(dec_target_batch)
print()

src_batch shape: torch.Size([4, 8])
tensor([[16, 19, 16, 22,  0,  0,  0,  0],
        [14,  1,  8,  5,  5, 22,  0,  0],
        [ 3,  6,  5, 13, 22,  0,  0,  0],
        [ 2, 13, 12, 22,  0,  0,  0,  0]])

dec_input_batch torch.Size([4, 8])
tensor([[21, 16, 19, 16,  0,  0,  0,  0],
        [21,  5,  5,  8,  1, 14,  0,  0],
        [21, 13,  5,  6,  3,  0,  0,  0],
        [21, 12, 13,  2,  0,  0,  0,  0]])

dec_target_batch torch.Size([4, 8])
tensor([[16, 19, 16, 22,  0,  0,  0,  0],
        [ 5,  5,  8,  1, 14, 22,  0,  0],
        [13,  5,  6,  3, 22,  0,  0,  0],
        [12, 13,  2, 22,  0,  0,  0,  0]])



In [27]:
# Position Encoding
class PositionalEncoding(nn.Module):
    def __init__(self,d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len,d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:,0::2] = torch.sin(position * div_term)         # 짝수
        pe[:,1::2] = torch.cos(position * div_term)         # 홀수

        pe = pe.unsqueeze(0)
        self.register_buffer("pe",pe)
    def forward(self,x):
        seq_len = x.size(1)
        return x + self.pe[:,:seq_len,:]

In [28]:
# Transformer 모델 만들기(작은버전)
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, d_ff=128, max_len=100):
        super().__init__()

        assert d_model % num_heads ==0

        self.d_model = d_model

        self.src_embed = nn.Embedding(vocab_size,d_model, padding_idx =PAD)
        self.tgt_embed = nn.Embedding(vocab_size,d_model, padding_idx =PAD)

        self.pos_encoder = PositionalEncoding(d_model, max_len = max_len)

        self.transformer = nn.Transformer(
            d_model = d_model,
            nhead = num_heads,
            num_encoder_layers = num_layers,
            num_decoder_layers = num_layers,
            dim_feedforward = d_ff,
            dropout=0.1,
            batch_first = True
        )
        
        self.fc_out = nn.Linear(d_model, vocab_size)
    def make_pad_mask(self,x):
        return (x==PAD)
    def forward(self,src, tgt):
        src_pad_mask = self.make_pad_mask(src)
        tgt_pad_mask = self.make_pad_mask(tgt)

        tgt_len = tgt.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len).to(src.device)

        src_emb = self.src_embed(src) * math.sqrt(self.d_model)
        tgt_emb = self.tgt_embed(tgt) * math.sqrt(self.d_model)
        #입력 단계(encoding): "단어 의미가 위치 정보에 묻히지 않게 키우자" -> 곱하기(신호 증폭)
        # ex) d_model=512이면 약 22.6을 곱해주는 것.
        #연산 단계(Attention): "값이 너무 커져서 학습이 안되는 것을 막기 위해 줄이자" -> 나누기

        src_emb = self.pos_encoder(src_emb)
        tgt_emb = self.pos_encoder(tgt_emb)

        out = self.transformer(
            src = src_emb,
            tgt = tgt_emb,
            src_key_padding_mask = src_pad_mask,
            tgt_key_padding_mask = tgt_pad_mask,
            memory_key_padding_mask = src_pad_mask
        )

        logits = self.fc_out(out)
        return logits

In [29]:
# 모델 생성
model = SimpleTransformer(
    vocab_size = vocab_size,
    d_model = 64,
    num_heads=4,
    num_layers=2,
    d_ff = 128,
    max_len = max_len
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

SimpleTransformer(
  (src_embed): Embedding(23, 64, padding_idx=0)
  (tgt_embed): Embedding(23, 64, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=128, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=128, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): Tran

In [30]:
# forward shape 확인
src_batch, dec_input_batch, dec_target_batch = make_batch(batch_size=4)

logits = model(src_batch, dec_input_batch)

print("src_batch shape:", src_batch.shape)
print("dec_input_batch:", dec_input_batch.shape)
print("logits shape:", logits.shape)
# 각 위치마다 23개 토큰 중 하나를 예측

src_batch shape: torch.Size([4, 8])
dec_input_batch: torch.Size([4, 8])
logits shape: torch.Size([4, 8, 23])


In [31]:
# 추론
## greedy decoding 함수

def greedy_decoding(model, src_seq, max_len=8):
    # model : 학습된 Transformer
    # src_seq : 입력
    # max_len: 생성할 문장의 최대 길이
    model.eval() # 모델이 평가 모드로 전환(Dropout, Batch Normalization 비활성화)

    src = torch.tensor([src_seq + [EOS]], dtype=torch.long, device=device)
    # 입력 시퀀스 EOS(End of Sentence) 토큰을 추가하고 모델 입력용 텐서로 변환.
    ## [batch_size=1, seq_len]형태의 2차원 텐서가 됨.
    
    generated = [BOS]
    # 생성될 문장의 시작을 알리는 BOS(Begging of Sentence) 토큰으로 초기화

    for _ in range(max_len):
        tgt = torch.tensor([generated], dtype=torch.long, device=device)
        #지금까지 생성된 토큰 리스트를 모델의 타겟 입력(tgt) 텐서로 변환

        with torch.no_grad(): #추론 단계이므로 기울기 계산을 생략하여 메모리 절약
            logits = model(src,tgt)
            # 모델에 입력(src)과 지금까지 생성된 target(tgt)을 넣어 예측값을 계산
            # 출력 shape: [1, 현재_tgt_길이, vocab_size]

        next_token = logits[0,-1].argmax().item()
        # 모델 출력의 마지막 시점(-1)의 로짓(logits)값에서 가장 높은 확률을 가진 인덱스를 선택
        # [0,-1]은 첫번째 배치(0)의 마지막 단어(-1)를 의미함.
        generated.append(next_token)
        # 선택된 토큰을 생성리스트에 추가

        if next_token == EOS:
            break
        # 만약 모델이 문장의 끝을 알리는 EOS 토큰을 예측했다면 생성을 중단
    return generated
    #최종적으로 생성된 토큰 리스트 반환
    
    
# 학습 전 예측 보기
print("학습 전 예측")
print("="*50)

for _ in range(5):
    src, tgt = generate_sample()
    pred = greedy_decoding(model,src,max_len = max_len)

    pred_tokens = pred[1:] #BOS 제거
    if EOS in pred_tokens:
        pred_tokens = pred_tokens[:pred_tokens.index(EOS)]

    print("입력:", src)
    print("출력:",tgt)
    print("예측",pred_tokens)
    print("-" *50)
    
    
# loss 계산
src_batch, dec_input_batch, dec_target_batch = make_batch(batch_size=8)

logits = model(src_batch, dec_input_batch)

loss = criterion(
    logits.reshape(-1,vocab_size),
    dec_target_batch.reshape(-1)
)

print("초기 loss:", loss.item())

학습 전 예측
입력: [11, 5, 20]
출력: [20, 5, 11]
예측 [9, 9, 9, 9, 9, 9, 9, 9]
--------------------------------------------------
입력: [20, 3, 8, 1, 14]
출력: [14, 1, 8, 3, 20]
예측 []
--------------------------------------------------
입력: [11, 19, 19, 3]
출력: [3, 19, 19, 11]
예측 []
--------------------------------------------------
입력: [13, 18, 3]
출력: [3, 18, 13]
예측 []
--------------------------------------------------
입력: [2, 19, 17, 6, 16]
출력: [16, 6, 17, 19, 2]
예측 []
--------------------------------------------------
초기 loss: 3.2699687480926514


In [ ]:

# 학습 후

num_epochs=1000
for epoch in range(num_epochs):
    model.train()

    src_batch, dec_input_batch, dec_target_batch = make_batch(batch_size=64)

    logits = model(src_batch, dec_input_batch)

    loss = criterion(
        logits.reshape(-1,vocab_size),
        dec_target_batch.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    #optimizer.step #참조 -> 함수가 실행되지 않음.
    optimizer.step() #함수가 실행 -> gradient 방향으로 학습을 진행하여라.

    if (epoch+1) % 100 ==0:
        print(f"Epoch [{epoch+1} / {num_epochs}] Loss: {loss.item():.4f}")
# 완전히 새로운 테스트 데이터로 일반화 확인
test_data = [
    [4,1,7],
    [2,9,5],
    [8,3,6,1],
    [10,2,14,7,5],
    [19,4,8]
]

print("처음 보는 데이터")
print("="*50)

for src in test_data:
    expected = list(reversed(src))
    pred = greedy_decoding(model, src, max_len = max_len)
    
    pred_tokens = pred[1:] #BOS 제거
    if EOS in pred_tokens:
        pred_tokens = pred_tokens[:pred_tokens.index(EOS)]

    print("입력:", src)
    print("출력:",tgt)
    print("예측",pred_tokens)
    print("일치", pred_tokens ==expected)
    print("-" *50)
        

Epoch [100 / 1000] Loss: 1.7277
Epoch [200 / 1000] Loss: 1.6279
Epoch [300 / 1000] Loss: 1.5925
Epoch [400 / 1000] Loss: 1.5442
Epoch [500 / 1000] Loss: 1.2416
Epoch [600 / 1000] Loss: 0.6887
Epoch [700 / 1000] Loss: 0.5068
Epoch [800 / 1000] Loss: 0.2995
Epoch [900 / 1000] Loss: 0.3967
Epoch [1000 / 1000] Loss: 0.2928


In [ ]:
def evalute_accuracy(model, num_test=100):
    model.eval()
    correct=0

    for _ in range(num_test):
        src, tgt = generate_sample()
        pred = greedy_decoding(model, src, max_len=max_len)

        pred_tokens = pred[1:]
        if EOS in pred_tokens:
            pred_tokens = pred_tokens[:pred_tokens.index(EOS)]

        if pred_tokens == tgt:
            correct +=1
    return correct / num_test

acc = evalute_accuracy(model, num_test=100)
print(f"테스트 정확도: {acc:.2%}") 